# CV Parser – Kaggle GPU Backend Server

This notebook:
1. Installs required packages
2. Loads **Mistral-Nemo-Instruct-2407** on the Kaggle T4 GPU
3. Starts a **Flask** REST API (`/parse` endpoint)
4. Tunnels the API to the public internet via **ngrok**

Copy the printed ngrok URL and paste it into your Streamlit app.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q flask pyngrok pypdf transformers==4.52.4 accelerate

In [ ]:
# ── 2. Load the model ────────────────────────────────────────────────────────
import os
import re
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model weights (this takes ~5 min on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Model ready on:", model.device)

In [ ]:
# ── 3. Define inference helpers ──────────────────────────────────────────────
# No LangChain dependency – plain prompt + regex JSON extraction,
# identical to the original CV_Parser.ipynb approach.

CV_TEMPLATE = """\
You are a smart assistant that extracts information from CVs/resumes.

Extract the following fields and respond with ONLY a valid JSON object \
(no markdown, no code fences, no extra text):

{{
    "FullName":   "candidate's full name",
    "Email":      "candidate's email address",
    "Phone":      "candidate's phone number",
    "Education":  "education history with degree, institution and year",
    "Skills":     ["skill1", "skill2"],
    "Experience": ["role, company, duration"],
    "Projects":   ["project name and brief description (empty list if none found)"]
}}

CV Text:
{cv_text}

Remember: output ONLY the JSON object, nothing else.
"""


def generate_text(prompt: str, max_new_tokens: int = 1024) -> str:
    """Run a single forward pass and return only the newly generated tokens."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def parse_cv_text(cv_text: str) -> dict:
    """Full pipeline: build prompt → run model → extract JSON → return dict."""
    prompt = CV_TEMPLATE.format(cv_text=cv_text)
    raw_response = generate_text(prompt)

    # 1) Try a ```json ... ``` code block first
    code_block = re.findall(r'```json\s*(.*?)\s*```', raw_response, re.DOTALL)
    if code_block:
        candidate = code_block[-1]
    else:
        # 2) Fall back to the first {...} in the response
        brace_match = re.search(r'\{.*\}', raw_response, re.DOTALL)
        candidate = brace_match.group(0) if brace_match else ""

    try:
        return json.loads(candidate)
    except (json.JSONDecodeError, ValueError):
        return {
            "FullName":   "Could not extract",
            "Email":      "Could not extract",
            "Phone":      "Could not extract",
            "Education":  "Could not extract",
            "Skills":     [],
            "Experience": [],
            "_raw":        raw_response,
        }


print("Inference helpers ready.")

In [ ]:
# ── 4. Flask API ─────────────────────────────────────────────────────────────
from flask import Flask, request, jsonify
from pypdf import PdfReader
import io

app = Flask(__name__)


@app.route("/health", methods=["GET"])
def health():
    """Quick liveness check – Streamlit can ping this to confirm the server is up."""
    return jsonify({"status": "ok", "model": MODEL_NAME})


@app.route("/parse", methods=["POST"])
def parse_endpoint():
    """
    Accepts either:
      • multipart/form-data  with a field named 'file'  (PDF upload)
      • application/json     with a field named 'text'  (raw CV text)
    Returns JSON with the extracted CV fields.
    """
    try:
        # --- PDF upload path ---
        if 'file' in request.files:
            pdf_bytes = request.files['file'].read()
            reader = PdfReader(io.BytesIO(pdf_bytes))
            cv_text = "".join(page.extract_text() or "" for page in reader.pages)

        # --- Raw text path ---
        elif request.is_json and 'text' in request.json:
            cv_text = request.json['text']

        else:
            return jsonify({"error": "Send a PDF file (field: 'file') or JSON with field 'text'"}), 400

        if not cv_text.strip():
            return jsonify({"error": "Could not extract any text from the provided input"}), 422

        result = parse_cv_text(cv_text)
        return jsonify(result)

    except Exception as exc:
        return jsonify({"error": str(exc)}), 500


print("Flask app defined.")

In [ ]:
# ── 5. Start ngrok tunnel and run Flask ──────────────────────────────────────
from pyngrok import ngrok, conf
import threading

NGROK_AUTH_TOKEN = "" #Put your ngrok api key here
FLASK_PORT = 5000

# Authenticate ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any leftover tunnels from a previous run
ngrok.kill()

# Open tunnel
tunnel = ngrok.connect(FLASK_PORT, "http")
public_url = tunnel.public_url

print("=" * 60)
print(f"  ngrok tunnel active!")
print(f"  Public URL : {public_url}")
print(f"  Health     : {public_url}/health")
print(f"  Parse API  : {public_url}/parse")
print("=" * 60)
print("Paste the Public URL into your Streamlit app and click 'Connect'.")

# Run Flask in a background thread so the cell stays non-blocking
flask_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=FLASK_PORT, use_reloader=False),
    daemon=True
)
flask_thread.start()
print("Flask server running in background thread.")

In [ ]:
# ── 6. (Optional) Keep the notebook alive ────────────────────────────────────
# Kaggle kernels idle-out after ~60 min of no output.
# Running this cell keeps a heartbeat going so the server stays up.
import time

print("Keeping notebook alive. Interrupt the kernel to stop the server.")
while True:
    time.sleep(300)   # print a dot every 5 minutes
    print(".", end="", flush=True)